# Phase 3 – Modélisation statistique (Régression)



## Définition des variables

La variable à expliquer (Y) est le taux d’engagement.

Les variables explicatives (X) retenues pour la modélisation sont :
- le nombre de hashtags
- la longueur de la légende
- le nombre d’impressions


In [14]:
import pandas as pd

df = pd.read_csv("/content/Instagram data.csv", encoding="latin1")
df.head()


,Impressions,From Home,From Hashtags,From Explore,From Other,Saves,Comments,Shares,Likes,Profile Visits,Follows,Caption,Hashtags
0,3920,2586,1028,619,56,98,9,5,162,35,2,Here are some of the most important data visua...,#finance #money #business #investing #investme...
1,5394,2727,1838,1174,78,194,7,14,224,48,10,Here are some of the best data science project...,#healthcare #health #covid #data #datascience ...
2,4021,2085,1188,0,533,41,11,1,131,62,12,Learn how to train a machine learning model an...,#data #datascience #dataanalysis #dataanalytic...
3,4528,2700,621,932,73,172,10,7,213,23,8,Heres how you can write a Python program to d...,#python #pythonprogramming #pythonprojects #py...
4,2518,1704,255,279,37,96,5,4,123,8,0,Plotting annotations while visualizing your da...,#datavisualization #datascience #data #dataana...


In [15]:
df["caption_length"] = df["Caption"].apply(len)
df["hashtags_count"] = df["Hashtags"].apply(lambda x: len(str(x).split()))
df["engagement"] = (df["Likes"] + df["Comments"] + df["Shares"]) / df["Impressions"]


In [16]:
X = df[["hashtags_count", "caption_length", "Impressions"]]
y = df["engagement"]


## Modèle de régression linéaire

Nous utilisons une régression linéaire afin d’expliquer le taux
d’engagement en fonction des variables sélectionnées.


In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model_reg = LinearRegression()
model_reg.fit(X, y)

y_pred = model_reg.predict(X)


In [18]:
r2 = r2_score(y, y_pred)
r2


0.34914874336334345

Le modèle de régression linéaire atteint un coefficient de détermination
R2 modéré, ce qui montre qu’une partie de la variabilité du taux d’engagement est expliquée par ces trois variables, mais qu’il existe d’autres facteurs non observés (contenu exact, visuels, contexte, etc.)

### Limites du modèle

Le modèle de régression linéaire utilisé reste volontairement simple.
Il permet d’identifier des tendances générales, mais ne capture pas
nécessairement toutes les relations complexes ou non linéaires entre
les variables et le taux d’engagement.


In [19]:
coefficients = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": model_reg.coef_
})

coefficients


,Variable,Coefficient
0,hashtags_count,2.621517e-04
1,caption_length,-1.377230e-05
2,Impressions,-9.688852e-07


## Interprétation des résultats

Le coefficient de détermination R² permet d’évaluer la qualité du
modèle. Les coefficients indiquent l’influence de chaque variable
sur le taux d’engagement, toutes choses égales par ailleurs.

On observe que certaines variables ont un impact plus important
que d’autres sur l’engagement des publications.


# Phase 4 – Modélisation statistique (Classification)




## Création de la variable cible binaire

Un post est considéré comme ayant un fort engagement si son taux
d’engagement est supérieur à la médiane.


In [20]:
median_engagement = df["engagement"].median()
df["high_engagement"] = (df["engagement"] > median_engagement).astype(int)

df["high_engagement"].value_counts()


,count
high_engagement,
0,60
1,59


## Séparation des données

Les données sont séparées en un jeu d’entraînement et un jeu de test
afin d’évaluer les performances du modèle.


In [21]:
from sklearn.model_selection import train_test_split

X_clf = df[["hashtags_count", "caption_length", "Impressions"]]
y_clf = df["high_engagement"]

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42
)


## Modèle de régression logistique

Nous utilisons une régression logistique pour prédire si un post
atteindra un fort niveau d’engagement.


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

model_log = LogisticRegression(max_iter=1000)
model_log.fit(X_train, y_train)

y_pred_log = model_log.predict(X_test)


In [23]:
confusion_matrix(y_test, y_pred_log)


array([[14,  4],
       [ 4, 14]])

In [24]:
print(classification_report(y_test, y_pred_log))


              precision    recall  f1-score   support

           0       0.78      0.78      0.78        18
           1       0.78      0.78      0.78        18

    accuracy                           0.78        36
   macro avg       0.78      0.78      0.78        36
weighted avg       0.78      0.78      0.78        36



### Interprétation des résultats de la régression logistique

La régression logistique permet de prédire si un post atteindra un fort
niveau d’engagement à partir de ses caractéristiques. Les résultats
montrent que le modèle est capable de distinguer les posts à fort et
faible engagement, bien que certaines erreurs de classification
subsistent.


## Modèle alternatif : Arbre de décision

Un arbre de décision est utilisé afin de comparer les performances
avec la régression logistique.


In [25]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)


In [26]:
confusion_matrix(y_test, y_pred_tree)


array([[14,  4],
       [ 4, 14]])